# Test Driven Development

In [ ]:
%load_ext autoreload
%load_ext sql
%autoreload 2

import sqlite3

import matplotlib.pyplot as plt
import pandas as pd
from config import settings

## Building Our Data Module

**For our application, we're going to keep all the classes we use to extract, transform, and load data in a single module that we'll call data.**

## AlphaVantage API Class

Let's get started by taking the code we created in the last lesson and incorporating it into a class that will be in charge of getting data from the AlphaVantage API.



In the data module, create a class definition for AlphaVantageAPI. For now, making sure that it has an __init__ method that attaches your 
API key as the attribute __api_key. Once you're done, import the class below and create an instance of it called av.

In [ ]:
# Import `AlphaVantageAPI`
from data import AlphaVantageAPI

# Create instance of `AlphaVantageAPI` class
av = AlphaVantageAPI()

print("av type:", type(av))

Create a get_daily method for your AlphaVantageAPI class. Once you're done, use the cell below to fetch the stock data for the renewable energy company Suzlon and assign it to the DataFrame df_suzlon.

In [ ]:
# Define Suzlon ticker symbol
ticker = "SUZLON.BSE"

# Use your `av` object to get daily data
df_suzlon = av.get_daily(ticker=ticker)

print("df_suzlon type:", type(df_suzlon))
print("df_suzlon shape:", df_suzlon.shape)
df_suzlon.head()

In [ ]:
"""This is for all the code used to interact with the AlphaVantage API
and the SQLite database. Remember that the API relies on a key that is
stored in your `.env` file and imported via the `config` module.
"""

import sqlite3

import pandas as pd
import requests
from config import settings


class AlphaVantageAPI:
    def __init__(self, api_key=settings.alpha_api_key):

        self.__api_key = api_key

    def get_daily(self, ticker, output_size="full"): 
        """Get daily time series of an equity from AlphaVantage API.
    
        Parameters
        ----------
        ticker : str
            The ticker symbol of the equity.
        output_size : str, optional
            Number of observations to retrieve. "compact" returns the
            latest 100 observations. "full" returns all observations for
            equity. By default "full".
    
        Returns
        -------
        pd.DataFrame
            Columns are 'open', 'high', 'low', 'close', and 'volume'.
            All are numeric.
        """
        # Create URL (8.1.5)
        url = (
            "https://learn-api.wqu.edu/1/data-services/alpha-vantage/query?"
            "function=TIME_SERIES_DAILY&"
            f"symbol={ticker}&"
            f"outputsize={output_size}&"
            f"datatype=json&"
            f"apikey={self.__api_key}"
        )
    
        # Send request to API (8.1.6)
        response = requests.get(url=url)
    
        # Extract JSON data from response (8.1.10)
        response_data = response.json()
        
        # Check if there's been an error
        if "Time Series (Daily)" not in response_data.keys():
            raise Exception(
            f"Invalid API call. Check that ticker symbol '{ticker}' is correct."
        )
        
        # Read data into DataFrame (8.1.12 & 8.1.13)
        stock_data = response_data["Time Series (Daily)"]
        df = pd.DataFrame.from_dict(stock_data, orient="index", dtype=float)
    
    
        # Convert index to `DatetimeIndex` named "date" (8.1.14)
        df.index = pd.to_datetime(df.index)
        df.index.name = "date"
    
        # Remove numbering from columns (8.1.15)
        df.columns = [c.split(". ")[1] for c in df.columns]
        
        # Return DataFrame
        return df
        
class SQLRepository:
    def __init__(self, connection):
        self.connection = connection
        pass

    def insert_table():
        """Insert DataFrame into SQLite database as table

        Parameters
        ----------
        table_name : str
        records : pd.DataFrame
        if_exists : str, optional
            How to behave if the table already exists.

            - 'fail': Raise a ValueError.
            - 'replace': Drop the table before inserting new values.
            - 'append': Insert new values to the existing table.

            Dafault: 'fail'

        Returns
        -------
        dict
            Dictionary has two keys:

            - 'transaction_successful', followed by bool
            - 'records_inserted', followed by int
        """
        pass

    def read_table():
        """Read table from database.

        Parameters
        ----------
        table_name : str
            Name of table in SQLite database.
        limit : int, None, optional
            Number of most recent records to retrieve. If `None`, all
            records are retrieved. By default, `None`.

        Returns
        -------
        pd.DataFrame
            Index is DatetimeIndex "date". Columns are 'open', 'high',
            'low', 'close', and 'volume'. All columns are numeric.
        """
        # Create SQL query (with optional limit)
        

        # Retrieve data, read into DataFrame
        

        # Return DataFrame
    
        pass


In [ ]:
## Create four assert statements to test the output of your get_daily method. Use the comments below as a guide.

# Does `get_daily` return a DataFrame?
assert isinstance(df_suzlon, pd.DataFrame)

# Does DataFrame have 5 columns?
assert df_suzlon.shape[1] == 5

# Does DataFrame have a DatetimeIndex?
assert isinstance(df_suzlon.index, pd.DatetimeIndex)

# Is the index name "date"?
assert df_suzlon.index.name == "date"

In [ ]:
## Create two more tests for the output of your get_daily method.

# Does DataFrame have correct column names?
assert df_suzlon.columns.to_list() == ['open', 'high', 'low', 'close', 'volume']

# Are columns correct data type?
assert all(df_suzlon.dtypes == float)

# SQL Repository Class

It wouldn't be efficient if our application needed to get data from the AlphaVantage API every time we wanted to explore our data or build a model, so we'll need to store our data in a database. Because our data is highly structured (each DataFrame we extract from AlphaVantage is always going to have the same five columns), it makes sense to use a SQL database.

We'll use SQLite for our database. For consistency, this database will always have the same name, which we've stored in our .env file.

In [ ]:
## Connect to the database whose name is stored in the .env file for this project. Be sure to set the check_same_thread argument to False. 
# Assign the connection to the variable connection.

connection = sqlite3.connect(database=settings.db_name, check_same_thread=False)

print("connection type:", type(connection))

We've got a connection, and now we need to start building the class that will handle all our transactions with the database. With this class, though, we're going to create our tests before writing the class definition.

In [ ]:
## Write two tests for the SQLRepository class, using the comments below as a guide.

# Import class definition
from data import SQLRepository

# Create instance of class
repo = SQLRepository(connection=connection)

# Does `repo` have a "connection" attribute?
assert hasattr(repo, "connection")

# Is the "connection" attribute a SQLite `Connection`?
assert isinstance(repo.connection, sqlite3.Connection)

*The next method we need for the SQLRepository class is one that allows us to store information. In SQL talk, this is generally referred to as inserting tables into the database.*

Add an insert_table method to your SQLRepository class. As a guide use the assert statements below and the docstring in the data module. When you're done, run the cell below to check your work.

In [ ]:
"""This is for all the code used to interact with the AlphaVantage API
and the SQLite database. Remember that the API relies on a key that is
stored in your `.env` file and imported via the `config` module.
"""

import sqlite3

import pandas as pd
import requests
from config import settings


class AlphaVantageAPI:
    def __init__(self, api_key=settings.alpha_api_key):

        self.__api_key = api_key

    def get_daily(self, ticker, output_size="full"): 
        """Get daily time series of an equity from AlphaVantage API.
    
        Parameters
        ----------
        ticker : str
            The ticker symbol of the equity.
        output_size : str, optional
            Number of observations to retrieve. "compact" returns the
            latest 100 observations. "full" returns all observations for
            equity. By default "full".
    
        Returns
        -------
        pd.DataFrame
            Columns are 'open', 'high', 'low', 'close', and 'volume'.
            All are numeric.
        """
        # Create URL (8.1.5)
        url = (
            "https://learn-api.wqu.edu/1/data-services/alpha-vantage/query?"
            "function=TIME_SERIES_DAILY&"
            f"symbol={ticker}&"
            f"outputsize={output_size}&"
            f"datatype=json&"
            f"apikey={self.__api_key}"
        )
    
        # Send request to API (8.1.6)
        response = requests.get(url=url)
    
        # Extract JSON data from response (8.1.10)
        response_data = response.json()
        
        # Check if there's been an error
        if "Time Series (Daily)" not in response_data.keys():
            raise Exception(
            f"Invalid API call. Check that ticker symbol '{ticker}' is correct."
        )
        
        # Read data into DataFrame (8.1.12 & 8.1.13)
        stock_data = response_data["Time Series (Daily)"]
        df = pd.DataFrame.from_dict(stock_data, orient="index", dtype=float)
    
    
        # Convert index to `DatetimeIndex` named "date" (8.1.14)
        df.index = pd.to_datetime(df.index)
        df.index.name = "date"
    
        # Remove numbering from columns (8.1.15)
        df.columns = [c.split(". ")[1] for c in df.columns]
        
        # Return DataFrame
        return df
        
class SQLRepository:
    def __init__(self, connection):
        self.connection = connection
        pass

    def insert_table(self, table_name, records, if_exists="fail"):
        """Insert DataFrame into SQLite database as table

        Parameters
        ----------
        table_name : str
        records : pd.DataFrame
        if_exists : str, optional
            How to behave if the table already exists.

            - 'fail': Raise a ValueError.
            - 'replace': Drop the table before inserting new values.
            - 'append': Insert new values to the existing table.

            Dafault: 'fail'

        Returns
        -------
        dict
            Dictionary has two keys:

            - 'transaction_successful', followed by bool
            - 'records_inserted', followed by int
        """
        n_inserted = records.to_sql(
            name=table_name, con=self.connection, if_exists=if_exists, index=True
        )
        
        return {
            "transaction_successful": True,
            "records_inserted": n_inserted
        }

    def read_table():
        """Read table from database.

        Parameters
        ----------
        table_name : str
            Name of table in SQLite database.
        limit : int, None, optional
            Number of most recent records to retrieve. If `None`, all
            records are retrieved. By default, `None`.

        Returns
        -------
        pd.DataFrame
            Index is DatetimeIndex "date". Columns are 'open', 'high',
            'low', 'close', and 'volume'. All columns are numeric.
        """
        # Create SQL query (with optional limit)
        

        # Retrieve data, read into DataFrame
        

        # Return DataFrame
    
        pass


In [ ]:
response = repo.insert_table(table_name=ticker, records=df_suzlon, if_exists="replace")

# Does your method return a dictionary?
assert isinstance(response, dict)

# Are the keys of that dictionary correct?
assert sorted(list(response.keys())) == ["records_inserted", "transaction_successful"]

In [ ]:
##  Write a SQL query to get the first five rows of the table of Suzlon data you just inserted into the database.

%sql sqlite:///stocks.sqlite
%%sql
SELECT *
FROM "SUZLON.BSE"
LIMIT 5

We can get insert data into our database, but let's not forget that we need to read data from it, too. Reading will be a little more complex than inserting, so let's start by writing code in this notebook before we incorporate it into our SQLRepository class.



In [ ]:
## First, write a SQL query to get all the Suzlon data. Then use pandas to extract the data from the database and read it into a DataFrame, 
## names df_suzlon_test

sql = "SELECT * FROM 'SUZLON.BSE'"
df_suzlon_test = pd.read_sql(
    sql=sql, con=connection, parse_dates=["date"], index_col="date"
)

print("df_suzlon_test type:", type(df_suzlon_test))
print()
print(df_suzlon_test.info())
df_suzlon_test.head()

Now that we know how to read a table from our database, let's turn our code into a proper function. But since we're doing backwards designs, we need to start with our tests.

In [ ]:
def read_table(table_name, limit=None):
    """Read table from database.

    Parameters
    ----------
    table_name : str
        Name of table in SQLite database.
    limit : int, None, optional
        Number of most recent records to retrieve. If `None`, all
        records are retrieved. By default, `None`.

    Returns
    -------
    pd.DataFrame
        Index is DatetimeIndex "date". Columns are 'open', 'high',
        'low', 'close', and 'volume'. All columns are numeric.
    """
    # Create SQL query (with optional limit)
    if limit:
        sql = f"SELECT * FROM '{table_name}' LIMIT {limit}"
    else:
        sql = f"SELECT * FROM '{table_name}'"
    

    # Retrieve data, read into DataFrame
    df = pd.read_sql(
        sql=sql, con=connection, parse_dates=["date"], index_col="date"
    )

    # Return DataFrame
    return df

In [ ]:
## Complete the assert statements below to test your read_table function. Use the comments as a guide.

# Assign `read_table` output to `df_suzlon`
df_suzlon = repo.read_table(table_name="SUZLON.BSE", limit=2500)  # noQA F821

# Is `df_suzlon` a DataFrame?
assert isinstance(df_suzlon, pd.DataFrame)

# Does it have a `DatetimeIndex`?
assert isinstance(df_suzlon.index, pd.DatetimeIndex)

# Is the index named "date"?
assert df_suzlon.index.name == "date"

# Does it have 2,500 rows and 5 columns?
assert df_suzlon.shape == (2500, 5)

# Are the column names correct?
assert df_suzlon.columns.to_list () == ['open', 'high', 'low', 'close', 'volume']

# Are the column data types correct?
assert all(df_suzlon.dtypes == float)

# Print `df_suzlon` info
print("df_suzlon shape:", df_suzlon.shape)
print()
print(df_suzlon.info())
df_suzlon.head()

In [ ]:
## Turn the read_table function into a method for your SQLRepository class.

"""This is for all the code used to interact with the AlphaVantage API
and the SQLite database. Remember that the API relies on a key that is
stored in your `.env` file and imported via the `config` module.
"""

import sqlite3

import pandas as pd
import requests
from config import settings


class AlphaVantageAPI:
    def __init__(self, api_key=settings.alpha_api_key):

        self.__api_key = api_key

    def get_daily(self, ticker, output_size="full"): 
        """Get daily time series of an equity from AlphaVantage API.
    
        Parameters
        ----------
        ticker : str
            The ticker symbol of the equity.
        output_size : str, optional
            Number of observations to retrieve. "compact" returns the
            latest 100 observations. "full" returns all observations for
            equity. By default "full".
    
        Returns
        -------
        pd.DataFrame
            Columns are 'open', 'high', 'low', 'close', and 'volume'.
            All are numeric.
        """
        # Create URL (8.1.5)
        url = (
            "https://learn-api.wqu.edu/1/data-services/alpha-vantage/query?"
            "function=TIME_SERIES_DAILY&"
            f"symbol={ticker}&"
            f"outputsize={output_size}&"
            f"datatype=json&"
            f"apikey={self.__api_key}"
        )
    
        # Send request to API (8.1.6)
        response = requests.get(url=url)
    
        # Extract JSON data from response (8.1.10)
        response_data = response.json()
        
        # Check if there's been an error
        if "Time Series (Daily)" not in response_data.keys():
            raise Exception(
            f"Invalid API call. Check that ticker symbol '{ticker}' is correct."
        )
        
        # Read data into DataFrame (8.1.12 & 8.1.13)
        stock_data = response_data["Time Series (Daily)"]
        df = pd.DataFrame.from_dict(stock_data, orient="index", dtype=float)
    
    
        # Convert index to `DatetimeIndex` named "date" (8.1.14)
        df.index = pd.to_datetime(df.index)
        df.index.name = "date"
    
        # Remove numbering from columns (8.1.15)
        df.columns = [c.split(". ")[1] for c in df.columns]
        
        # Return DataFrame
        return df
        
class SQLRepository:
    def __init__(self, connection):
        self.connection = connection
        pass

    def insert_table(self, table_name, records, if_exists="fail"):
        """Insert DataFrame into SQLite database as table

        Parameters
        ----------
        table_name : str
        records : pd.DataFrame
        if_exists : str, optional
            How to behave if the table already exists.

            - 'fail': Raise a ValueError.
            - 'replace': Drop the table before inserting new values.
            - 'append': Insert new values to the existing table.

            Dafault: 'fail'

        Returns
        -------
        dict
            Dictionary has two keys:

            - 'transaction_successful', followed by bool
            - 'records_inserted', followed by int
        """
        n_inserted = records.to_sql(
            name=table_name, con=self.connection, if_exists=if_exists, index=True
        )
        
        return {
            "transaction_successful": True,
            "records_inserted": n_inserted
        }

    def read_table(self, table_name, limit=None):
        """Read table from database.
    
        Parameters
        ----------
        table_name : str
            Name of table in SQLite database.
        limit : int, None, optional
            Number of most recent records to retrieve. If `None`, all
            records are retrieved. By default, `None`.
    
        Returns
        -------
        pd.DataFrame
            Index is DatetimeIndex "date". Columns are 'open', 'high',
            'low', 'close', and 'volume'. All columns are numeric.
        """
        # Create SQL query (with optional limit)
        if limit:
            sql = f"SELECT * FROM '{table_name}' LIMIT {limit}"
        else:
            sql = f"SELECT * FROM '{table_name}'"
        
    
        # Retrieve data, read into DataFrame
        df = pd.read_sql(
            sql=sql, con=self.connection, parse_dates=["date"], index_col="date"
        )
    
        # Return DataFrame
        return df


# Comparing Stock Returns

We already have the data for Suzlon Energy in our database, but we need to add the data for Ambuja Cement before we can compare the two stocks.


Use the instances of the AlphaVantageAPI and SQLRepository classes you created in this lesson (av and repo, respectively) to get the stock data for Ambuja Cement and read it into the database.

In [ ]:
ticker = "AMBUJACEM.BSE"

# Get Ambuja data using `av`
ambuja_records = av.get_daily(ticker=ticker)

# Insert `ambuja_records` database using `repo`
response = repo.insert_table(
    table_name=ticker, records=ambuja_records, if_exists="replace"
)

response

In [ ]:
## Using the read_table method you've added to your SQLRepository, extract the most recent 2,500 rows of data for Ambuja Cement from the database 
## and assign the result to df_ambuja.


ticker = "AMBUJACEM.BSE"
df_ambuja = repo.read_table(table_name=ticker, limit=2500)

print("df_ambuja type:", type(df_ambuja))
print("df_ambuja shape:", df_ambuja.shape)
df_ambuja.head()

We've spent a lot of time so far looking at this data, but what does it actually represent? It turns out the stock market is a lot like any other market: people buy and sell goods. The prices of those goods can go up or down depending on factors like supply and demand. In the case of a stock market, the goods being sold are stocks (also called equities or securities), which represent an ownership stake in a corporation.

During each trading day, the price of a stock will change, so when we're looking at whether a stock might be a good investment, we look at four types of numbers: open, high, low, close, volume. Open is exactly what it sounds like: the selling price of a share when the market opens for the day. Similarly, close is the selling price of a share when the market closes at the end of the day, and high and low are the respective maximum and minimum prices of a share over the course of the day. Volume is the number of shares of a given stock that have been bought and sold that day. Generally speaking, a firm whose shares have seen a high volume of trading will see more price variation of the course of the day than a firm whose shares have been more lightly traded.

Let's visualize how the price of Ambuja Cement changes over the last decade.

In [ ]:
## Plot the closing price of df_ambuja. Be sure to label your axes and include a legend.

fig, ax = plt.subplots(figsize=(15, 6))

# Plot `df_ambuja` closing price
df_ambuja["close"].plot(ax=ax, label="AMBUJACEM", color="C1")

# Label axes
plt.xlabel("Date")
plt.ylabel("Closing Price")

# Add legend
plt.legend()

In [ ]:
# Create a plot that shows the closing prices of df_suzlon and df_ambuja. Again, label your axes and include a legend.

fig, ax = plt.subplots(figsize=(15, 6))
# Plot `df_suzlon` and `df_ambuja`
df_suzlon["close"].plot(ax=ax, label="SUZLON")
df_ambuja["close"].plot(ax=ax, label="AMBUJACEM")

# Label axes
plt.xlabel("Date")
plt.ylabel("Closing Price")

# Add legend
plt.legend();

Looking at this plot, we might conclude that Ambuja Cement is a "better" stock than Suzlon energy because its price is higher. But price is just one factor that an investor must consider when creating an investment strategy. What is definitely true is that it's hard to do a head-to-head comparison of these two stocks because there's such a large price difference.

One way in which investors compare stocks is by looking at their returns instead. A return is the change in value in an investment, represented as a percentage. So let's look at the daily returns for our two stocks.

In [ ]:
## Add a "return" column to df_ambuja that shows the percentage change in the "close" column from one day to the next.

# Sort DataFrame ascending by date
df_ambuja.sort_index(ascending=True, inplace=True)

# Create "return" column
df_ambuja['return'] = df_ambuja['close'].pct_change() *100

print("df_ambuja shape:", df_ambuja.shape)
print(df_ambuja.info())
df_ambuja.head()

In [ ]:
## Add a "return" column to df_suzlon.


# Sort DataFrame ascending by date
df_suzlon.sort_index(ascending=True, inplace=True)

# Create "return" column
df_suzlon['return'] = df_suzlon['close'].pct_change() *100

print("df_suzlon shape:", df_suzlon.shape)
print(df_suzlon.info())
df_suzlon.head()

In [ ]:
## Now let's plot the returns for our two companies and see how the two compare.

## Plot the returns for df_suzlon and df_ambuja. Be sure to label your axes and use legend.


fig, ax = plt.subplots(figsize=(15, 6))
# Plot `df_suzlon` and `df_ambuja`
df_suzlon["return"].plot(ax=ax, label="SUZLON")
df_ambuja["return"].plot(ax=ax, label="AMBUJACEM")

# Label axes
plt.xlabel("Date")
plt.ylabel("Daily Return")

# Add legend
plt.legend();